In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np
from src.data_loader import load_insurance_data
from src.hypothesis_tests import test_categorical_kpi, test_numerical_kpi

# Load your Version 2 cleaned data pulled via DVC
df = load_insurance_data('../data/MachineLearningRating_v3.txt')

# Create explicit target features needed for the KPIs
df['ClaimOccurred'] = np.where(df['TotalClaims'] > 0, 1, 0)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# Test1: Provinces (H0: No Risk Differences Across Provinces)

# Isolate data where an actual claim occurred
claims_df = df[df['TotalClaims'] > 0]

# Run numerical test
p_val_prov, t_stat_prov = test_numerical_kpi(claims_df, 'Province', 'Western Cape', 'Gauteng', 'TotalClaims')

print(f"--- Hypothesis 1: Province vs Claim Severity ---")
print(f"P-Value: {p_val_prov:.5f}")

# Automated Decision Engine
if p_val_prov < 0.05:
    print("Decision: REJECT the Null Hypothesis (H₀).")
    print("Status: There is a statistically significant difference in claim severity between provinces.")
else:
    print("Decision: FAIL TO REJECT the Null Hypothesis (H₀).")
    print("Status: No statistically significant difference in claim severity detected.")

--- Hypothesis 1: Province vs Claim Severity ---
P-Value: 0.02551
Decision: REJECT the Null Hypothesis (H₀).
Status: There is a statistically significant difference in claim severity between provinces.


In [12]:
# Test2: Zip Codes Risk (H0: No Risk Differences Between Zip Codes)

# Run categorical test (Ensure 2000 and 8000 match your top found postal codes)
p_val_zip_freq, chi2_zip = test_categorical_kpi(df, 'PostalCode', 2000, 8000, 'ClaimOccurred')

print(f"--- Hypothesis 2: Zip Code vs Claim Frequency ---")
print(f"P-Value: {p_val_zip_freq:.5f}")

# Automated Decision Engine
if p_val_zip_freq < 0.05:
    print("Decision: REJECT the Null Hypothesis (H₀).")
    print("Status: There is a statistically significant difference in claim frequency between these zip codes.")
else:
    print("Decision: FAIL TO REJECT the Null Hypothesis (H₀).")
    print("Status: No statistically significant difference in claim frequency detected.")

--- Hypothesis 2: Zip Code vs Claim Frequency ---
P-Value: 0.71558
Decision: FAIL TO REJECT the Null Hypothesis (H₀).
Status: No statistically significant difference in claim frequency detected.


In [13]:
# Test3: Zip Codes Margin (H0: No Margin Differences Between Zip Codes)

# Run numerical test
p_val_zip_margin, t_stat_margin = test_numerical_kpi(df, 'PostalCode', 2000, 8000, 'Margin')

print(f"--- Hypothesis 3: Zip Code vs Net Margin ---")
print(f"P-Value: {p_val_zip_margin:.5f}")

# Automated Decision Engine
if p_val_zip_margin < 0.05:
    print("Decision: REJECT the Null Hypothesis (H₀).")
    print("Status: There is a statistically significant difference in financial margin between these zip codes.")
else:
    print("Decision: FAIL TO REJECT the Null Hypothesis (H₀).")
    print("Status: No statistically significant difference in financial margin detected.")

--- Hypothesis 3: Zip Code vs Net Margin ---
P-Value: 0.80702
Decision: FAIL TO REJECT the Null Hypothesis (H₀).
Status: No statistically significant difference in financial margin detected.


In [14]:
# Test4: Gender (H0: No Risk Difference Between Women and Men)

# Run categorical test
p_val_gender, chi2_gender = test_categorical_kpi(df, 'Gender', 'Female', 'Male', 'ClaimOccurred')

print(f"--- Hypothesis 4: Gender vs Claim Frequency ---")
print(f"P-Value: {p_val_gender:.5f}")

# Automated Decision Engine
if p_val_gender < 0.05:
    print("Decision: REJECT the Null Hypothesis (H₀).")
    print("Status: There is a statistically significant difference in claim frequency between Men and Women.")
else:
    print("Decision: FAIL TO REJECT the Null Hypothesis (H₀).")
    print("Status: No statistically significant difference in claim frequency detected.")

--- Hypothesis 4: Gender vs Claim Frequency ---
P-Value: 0.70614
Decision: FAIL TO REJECT the Null Hypothesis (H₀).
Status: No statistically significant difference in claim frequency detected.


## 📊 Hypothesis Testing Results Summary

The table below summarizes the statistical validations performed across our core performance indicators. Each test was evaluated against a strict significance threshold ($\alpha = 0.05$).

| Hypothesis ($H_0$) | Selected KPI | Statistical Test | P-Value | Decision ($\alpha = 0.05$) |
| :--- | :--- | :--- | :--- | :--- |
| **$H_0$ 1: Provinces** | Claim Severity | Independent Welch's $t$-test | 0.02551 | **REJECT $H_0$** |
| **$H_0$ 2: Zip Codes Risk** | Claim Frequency | Chi-Squared ($\chi^2$) of Independence | 0.71558 | **FAIL TO REJECT $H_0$** |
| **$H_0$ 3: Zip Codes Margin**| Net Margin | Independent Welch's $t$-test | 0.80702 | **FAIL TO REJECT $H_0$** |
| **$H_0$ 4: Gender Risk** | Claim Frequency | Chi-Squared ($\chi^2$) of Independence | 0.70614 | **FAIL TO REJECT $H_0$** |

## 🏢 Business Interpretations & Strategic Recommendations

### 1. Province vs. Claim Severity (Rejected $H_0$, $p = 0.0255$)
* **Statistical Interpretation:** We reject the null hypothesis. The test confirms a statistically significant difference in claim severity between Gauteng and the Western Cape. Gauteng does not just experience a higher volume of claims; the individual claims generated are structurally more expensive to settle.
* **Strategic Business Recommendation:** AlphaCare Insurance Solutions (ACIS) must implement a **regional risk adjustment factor** within its underwriting framework. Base premium pricing for policies written in Gauteng should be scaled upward to accommodate higher expected severity costs, protecting our underwriting margins from localized urban risk drivers.

### 2. Zip Codes Risk & Margin Performance (Failed to Reject $H_0$, $p > 0.05$)
* **Statistical Interpretation:** We fail to reject the null hypotheses for both Zip Code Claim Frequency ($p = 0.71558$) and Zip Code Net Margin ($p = 0.80702$). The mathematical variance between the tested high-volume postal codes is minor and fully attributable to random noise rather than systemic risk factors.
* **Strategic Business Recommendation:** This lack of statistical differentiation provides a clear pathway for geographic expansion. ACIS can standardize baseline premium requirements across these micro-regions and deploy **aggressive volume-acquisition marketing campaigns** in target postal codes without the immediate need to design hyper-localized geographic pricing penalties.

### 3. Gender vs. Claim Frequency (Failed to Reject $H_0$, $p = 0.7061$)
* **Statistical Interpretation:** We fail to reject the null hypothesis. There is no statistically significant difference in the historical proportion of claims filed by male vs. female policyholders within this dataset. Gender alone does not function as a reliable statistical indicator of an individual's likelihood to experience an accident.
* **Strategic Business Recommendation:** This represents a major opportunity to gain market share during our growth phase. Since female drivers do not carry hidden underwriting risks compared to male drivers, **ACIS can introduce highly competitive, gender-targeted premium discounts** to capture underrepresented demographic segments. This allows the business to steal low-risk market share from competitors who rely on traditional, unverified risk-tiering assumptions.